In [7]:
HYPERPARAM_ITERS = 50  # Number of iterations for hyperparameter tuning; TODO: increase this for better results, but it will take longer to run.
EXPORT = False  # Whether to export the trained models; set to True to save the models to disk.

# Data Preprocessing

While we did a lot of data cleaning and preprocessing in other notebooks (see notebooks in [data directory](./data)), we will load the data here and do some additional preprocessing to prepare it for model training. This includes combining flight and weather data, adding some historical and derived columns, as well as encoding categorical variables and splitting the data into training and testing sets.

In [8]:
import polars as pl

# Load the data
df = pl.read_parquet("data/flights/Full_Flight_Data.parquet")
df = df.sort("FL_DATE")
df.shape, df.head(), df.tail()

((480802, 60),
 shape: (5, 60)
 ┌─────────────┬─────────────┬────────────┬────────────┬───┬───────┬────────────┬────────────┬──────┐
 │ FL_DATE     ┆ MKT_UNIQUE_ ┆ MKT_CARRIE ┆ OP_UNIQUE_ ┆ … ┆ SPEED ┆ TC-DATA-SH ┆ TC-DATA-HO ┆      │
 │ ---         ┆ CARRIER     ┆ R_FL_NUM   ┆ CARRIER    ┆   ┆ ---   ┆ EET        ┆ LDER       ┆ ---  │
 │ datetime[μs ┆ ---         ┆ ---        ┆ ---        ┆   ┆ str   ┆ ---        ┆ ---        ┆ str  │
 │ ]           ┆ str         ┆ i64        ┆ str        ┆   ┆       ┆ str        ┆ str        ┆      │
 ╞═════════════╪═════════════╪════════════╪════════════╪═══╪═══════╪════════════╪════════════╪══════╡
 │ 2024-01-01  ┆ AA          ┆ 1064       ┆ AA         ┆ … ┆ 0000  ┆ A28NM      ┆ AIRBUS SAS ┆ null │
 │ 00:00:00    ┆             ┆            ┆            ┆   ┆       ┆            ┆ …          ┆      │
 │ 2024-01-01  ┆ AA          ┆ 1153       ┆ AA         ┆ … ┆ 0000  ┆            ┆ …          ┆ null │
 │ 00:00:00    ┆             ┆            ┆        

In [9]:
# Drop non-DCA originating flights
df = df.filter(pl.col("ORIGIN_AIRPORT_ID") == 11278)
df.shape, df.columns

((240391, 60),
 ['FL_DATE',
  'MKT_UNIQUE_CARRIER',
  'MKT_CARRIER_FL_NUM',
  'OP_UNIQUE_CARRIER',
  'OP_CARRIER_AIRLINE_ID',
  'TAIL_NUM',
  'OP_CARRIER_FL_NUM',
  'ORIGIN_AIRPORT_ID',
  'ORIGIN_CITY_MARKET_ID',
  'ORIGIN',
  'DEST_AIRPORT_ID',
  'CRS_DEP_TIME',
  'DEP_TIME',
  'DEP_DELAY',
  'DEP_DELAY_GROUP',
  'TAXI_OUT',
  'WHEELS_OFF',
  'WHEELS_ON',
  'TAXI_IN',
  'CRS_ARR_TIME',
  'ARR_TIME',
  'ARR_DELAY',
  'CANCELLED',
  'CANCELLATION_CODE',
  'DIVERTED',
  'CRS_ELAPSED_TIME',
  'ACTUAL_ELAPSED_TIME',
  'AIR_TIME',
  'FLIGHTS',
  'DISTANCE',
  'CARRIER_DELAY',
  'WEATHER_DELAY',
  'NAS_DELAY',
  'SECURITY_DELAY',
  'LATE_AIRCRAFT_DELAY',
  'FIRST_DEP_TIME',
  'DIV_AIRPORT_LANDINGS',
  'DIV_REACHED_DEST',
  'DIV_ACTUAL_ELAPSED_TIME',
  'N-NUMBER',
  'SERIAL NUMBER',
  'MFR MDL CODE',
  'ENG MFR MDL',
  'YEAR MFR',
  'N_NUMBER_CLEAN',
  'MFR_CODE_CLEAN',
  'CODE',
  'MFR',
  'MODEL',
  'TYPE-ACFT',
  'TYPE-ENG',
  'AC-CAT',
  'BUILD-CERT-IND',
  'NO-ENG',
  'NO-SEATS',
  'AC-W

In [10]:
# Bring in weather data
weather_df = pl.read_parquet("data/weather/isd_weather_data.parquet").sort("timestamp")

# Calculate the actual scheduled departure timestamp from the flight date and scheduled departure time. This keeps the weather alignment consistent with weather info that would have been available at the time of the scheduled departure
df = df.with_columns(
    (
        pl.col("FL_DATE")
        + pl.duration(
            hours=(pl.col("CRS_DEP_TIME") // 100).cast(pl.Int64),
            minutes=(pl.col("CRS_DEP_TIME") % 100).cast(pl.Int64),
        )
    ).alias("scheduled_departure_ts")
).sort("scheduled_departure_ts")

# Merge in most recent weather data that would have been known by scheduled departure
df = df.join_asof(
    weather_df,
    left_on="scheduled_departure_ts",
    right_on="timestamp",
    strategy="backward",
    tolerance="3h",
).drop("timestamp")
df.shape, df.head(), df.get_column("scheduled_departure_ts")

((240391, 73),
 shape: (5, 73)
 ┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
 │ FL_DATE   ┆ MKT_UNIQU ┆ MKT_CARRI ┆ OP_UNIQUE ┆ … ┆ precip-1h ┆ precip-6h ┆ visibilit ┆ ceiling_ │
 │ ---       ┆ E_CARRIER ┆ ER_FL_NUM ┆ _CARRIER  ┆   ┆ _trace    ┆ _trace    ┆ y_m       ┆ height_m │
 │ datetime[ ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
 │ μs]       ┆ str       ┆ i64       ┆ str       ┆   ┆ i32       ┆ i32       ┆ i32       ┆ i32      │
 ╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
 │ 2024-01-0 ┆ AA        ┆ 153       ┆ AA        ┆ … ┆ 0         ┆ 0         ┆ 16093     ┆ null     │
 │ 1         ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
 │ 00:00:00  ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
 │ 2024-01-0 ┆ AA        ┆ 555       ┆ AA        ┆ 

In [11]:
# TODO: Add historical/derived features

#compute rolling delay stats for each route
# Create route key
df = df.with_columns(
    pl.concat_str(
        pl.col("ORIGIN"), pl.lit("→"), pl.col("DEST_AIRPORT_ID").cast(pl.Utf8)
    ).alias("route")
)

# Add average delay per route
df = df.with_columns(
    pl.col("DEP_DELAY")
        .mean()
        .over("route")
        .alias("avg_rte_delay")
)

#compute number of departures in timeframes
df = df.sort("scheduled_departure_ts")

# Cumulative flight count by timestamp
cumsum = df.select("scheduled_departure_ts").with_columns(
    pl.int_range(1, pl.len() + 1).alias("cum_count")
)

for col_name, hours in [
    ("deps_last_1h",  1),
    ("deps_last_3h",  3),
    ("deps_last_6h",  6),
    ("deps_last_12h", 12),
    ("deps_last_24h", 24),
]:
    cutoff = pl.duration(hours=hours)

    # For each flight, find the cumulative count at (departure - window)
    lookback = df.select(
        (pl.col("scheduled_departure_ts") - cutoff).alias("ts_lookback")
    ).join_asof(
        cumsum.rename({"scheduled_departure_ts": "ts_lookback", "cum_count": "cum_at_lookback"}),
        on="ts_lookback",
        strategy="backward",
    ).fill_null(0)

    df = df.with_columns(
        (pl.int_range(1, pl.len() + 1) - pl.Series(lookback["cum_at_lookback"])).alias(col_name)
    )

#rolling mean delay, rolling cancellation rate for MKT_UNIQUE_CARRIER over 7d and 30d
df = df.sort("scheduled_departure_ts")

df = df.with_columns([
    (pl.col("DEP_DELAY") > 0).cast(pl.Float64).alias("_is_late"),
    pl.col("CANCELLED").cast(pl.Float64).alias("_cancelled"),
    pl.col("DEP_DELAY").cast(pl.Float64).alias("_dep_delay"),
])

for window, suffix in [("7d", "week"), ("30d", "month")]:
    agg = (
        df.rolling("scheduled_departure_ts", period=window, closed="left", group_by="MKT_UNIQUE_CARRIER")
        .agg([
            pl.col("_dep_delay").mean().alias(f"carrier_avg_delay_{suffix}"),
            pl.col("_cancelled").mean().alias(f"carrier_cancel_rate_{suffix}"),
        ])
    )
    df = df.join(
        agg[["scheduled_departure_ts", "MKT_UNIQUE_CARRIER",
             f"carrier_avg_delay_{suffix}",
             f"carrier_cancel_rate_{suffix}"]],
        on=["scheduled_departure_ts", "MKT_UNIQUE_CARRIER"],
        how="left"
    )

df = df.drop(["_is_late", "_cancelled", "_dep_delay"])

In [12]:
df = df.with_columns(
    pl.col(
        "MODEL"
    ).str.strip_chars()  # Remove leading/trailing whitespace from MODEL column (e.g., "737-800 " -> "737-800"
)
df.head()

FL_DATE,MKT_UNIQUE_CARRIER,MKT_CARRIER_FL_NUM,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,TAIL_NUM,OP_CARRIER_FL_NUM,ORIGIN_AIRPORT_ID,ORIGIN_CITY_MARKET_ID,ORIGIN,DEST_AIRPORT_ID,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DELAY_GROUP,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,CANCELLED,CANCELLATION_CODE,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,FLIGHTS,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,FIRST_DEP_TIME,DIV_AIRPORT_LANDINGS,…,MFR,MODEL,TYPE-ACFT,TYPE-ENG,AC-CAT,BUILD-CERT-IND,NO-ENG,NO-SEATS,AC-WEIGHT,SPEED,TC-DATA-SHEET,TC-DATA-HOLDER,,scheduled_departure_ts,temp,dewtemp,pressure,winddirection,windspeed,skycoverage,precipitation-1h,precipitation-6h,precip-1h_trace,precip-6h_trace,visibility_m,ceiling_height_m,route,avg_rte_delay,deps_last_1h,deps_last_3h,deps_last_6h,deps_last_12h,deps_last_24h,carrier_avg_delay_week,carrier_cancel_rate_week,carrier_avg_delay_month,carrier_cancel_rate_month
datetime[μs],str,i64,str,i64,str,i64,i64,i64,str,i64,i64,i64,f64,i64,f64,i64,i64,f64,i64,i64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,i64,…,str,str,str,str,str,str,str,str,str,str,str,str,str,datetime[μs],f64,f64,f64,f64,f64,str,f64,f64,i32,i32,i32,i32,str,f64,i64,i64,i64,i64,i64,f64,f64,f64,f64
2024-01-01 00:00:00,"""AA""",153,"""AA""",19805,"""N894NN""",153,11278,30852,"""DCA""",11057,500,456,-4.0,-1,10.0,506,605,6.0,638,611,-27.0,0.0,null,0.0,98.0,75.0,59.0,1.0,331.0,null,null,null,null,null,null,0,…,"""BOEING ""","""737-823""","""5""","""5 ""","""1""","""0""","""02""","""162""","""CLASS 3""","""0000""",""" """,""" …",null,2024-01-01 05:00:00,5.6,0.0,1015.7,0.0,0.0,"""2-3_oktas""",0.0,null,0,0,16093,null,"""DCA→11057""",18.570115,1,1,1,1,1,null,null,null,null
2024-01-01 00:00:00,"""AA""",153,"""AA""",19805,"""N894NN""",153,11278,30852,"""DCA""",11057,500,456,-4.0,-1,10.0,506,605,6.0,638,611,-27.0,0.0,null,0.0,98.0,75.0,59.0,1.0,331.0,null,null,null,null,null,null,0,…,"""BOEING ""","""737-823""","""5""","""5 ""","""1""","""0""","""02""","""162""","""CLASS 3""","""0000""",""" """,""" …",null,2024-01-01 05:00:00,5.6,0.0,1015.7,0.0,0.0,"""2-3_oktas""",0.0,null,0,0,16093,null,"""DCA→11057""",18.570115,1,1,1,1,1,null,null,null,null
2024-01-01 00:00:00,"""AA""",153,"""AA""",19805,"""N894NN""",153,11278,30852,"""DCA""",11057,500,456,-4.0,-1,10.0,506,605,6.0,638,611,-27.0,0.0,null,0.0,98.0,75.0,59.0,1.0,331.0,null,null,null,null,null,null,0,…,"""BOEING ""","""737-823""","""5""","""5 ""","""1""","""0""","""02""","""162""","""CLASS 3""","""0000""",""" """,""" …",null,2024-01-01 05:00:00,5.6,0.0,1015.7,0.0,0.0,"""2-3_oktas""",0.0,null,0,0,16093,null,"""DCA→11057""",18.570115,1,1,1,1,1,null,null,null,null
2024-01-01 00:00:00,"""AA""",153,"""AA""",19805,"""N894NN""",153,11278,30852,"""DCA""",11057,500,456,-4.0,-1,10.0,506,605,6.0,638,611,-27.0,0.0,null,0.0,98.0,75.0,59.0,1.0,331.0,null,null,null,null,null,null,0,…,"""BOEING ""","""737-823""","""5""","""5 ""","""1""","""0""","""02""","""162""","""CLASS 3""","""0000""",""" """,""" …",null,2024-01-01 05:00:00,5.6,0.0,1015.7,0.0,0.0,"""2-3_oktas""",0.0,null,0,0,16093,null,"""DCA→11057""",18.570115,1,1,1,1,1,null,null,null,null
2024-01-01 00:00:00,"""AA""",153,"""AA""",19805,"""N894NN""",153,11278,30852,"""DCA""",11057,500,456,-4.0,-1,10.0,506,605,6.0,638,611,-27.0,0.0,null,0.0,98.0,75.0,59.0,1.0,331.0,null,null,null,null,null,null,0,…,"""BOEING ""","""737-823""","""5""","""5 ""","""1""","""0""","""02""","""162""","""CLASS 3""","""0000""",""" """,""" …",null,2024-01-01 05:00:00,5.6,0.0,1015.7,0.0,0.0,"""2-3_oktas""",0.0,null,0,0,16093,null,"""DCA→11057""",18.570115,1,1,1,1,1,null,null,null,null


In [13]:
df = df.with_columns(
    pl.col(pl.Utf8).cast(pl.Categorical),
    pl.col("scheduled_departure_ts").dt.epoch(
        time_unit="s"
    ),  # Convert datetime to epoch seconds
)
df.head()

FL_DATE,MKT_UNIQUE_CARRIER,MKT_CARRIER_FL_NUM,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,TAIL_NUM,OP_CARRIER_FL_NUM,ORIGIN_AIRPORT_ID,ORIGIN_CITY_MARKET_ID,ORIGIN,DEST_AIRPORT_ID,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DELAY_GROUP,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,CANCELLED,CANCELLATION_CODE,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,FLIGHTS,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,FIRST_DEP_TIME,DIV_AIRPORT_LANDINGS,…,MFR,MODEL,TYPE-ACFT,TYPE-ENG,AC-CAT,BUILD-CERT-IND,NO-ENG,NO-SEATS,AC-WEIGHT,SPEED,TC-DATA-SHEET,TC-DATA-HOLDER,,scheduled_departure_ts,temp,dewtemp,pressure,winddirection,windspeed,skycoverage,precipitation-1h,precipitation-6h,precip-1h_trace,precip-6h_trace,visibility_m,ceiling_height_m,route,avg_rte_delay,deps_last_1h,deps_last_3h,deps_last_6h,deps_last_12h,deps_last_24h,carrier_avg_delay_week,carrier_cancel_rate_week,carrier_avg_delay_month,carrier_cancel_rate_month
datetime[μs],cat,i64,cat,i64,cat,i64,i64,i64,cat,i64,i64,i64,f64,i64,f64,i64,i64,f64,i64,i64,f64,f64,cat,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,cat,i64,…,cat,cat,cat,cat,cat,cat,cat,cat,cat,cat,cat,cat,cat,i64,f64,f64,f64,f64,f64,cat,f64,f64,i32,i32,i32,i32,cat,f64,i64,i64,i64,i64,i64,f64,f64,f64,f64
2024-01-01 00:00:00,"""AA""",153,"""AA""",19805,"""N894NN""",153,11278,30852,"""DCA""",11057,500,456,-4.0,-1,10.0,506,605,6.0,638,611,-27.0,0.0,null,0.0,98.0,75.0,59.0,1.0,331.0,null,null,null,null,null,null,0,…,"""BOEING ""","""737-823""","""5""","""5 ""","""1""","""0""","""02""","""162""","""CLASS 3""","""0000""",""" """,""" …",null,1704085200,5.6,0.0,1015.7,0.0,0.0,"""2-3_oktas""",0.0,null,0,0,16093,null,"""DCA→11057""",18.570115,1,1,1,1,1,null,null,null,null
2024-01-01 00:00:00,"""AA""",153,"""AA""",19805,"""N894NN""",153,11278,30852,"""DCA""",11057,500,456,-4.0,-1,10.0,506,605,6.0,638,611,-27.0,0.0,null,0.0,98.0,75.0,59.0,1.0,331.0,null,null,null,null,null,null,0,…,"""BOEING ""","""737-823""","""5""","""5 ""","""1""","""0""","""02""","""162""","""CLASS 3""","""0000""",""" """,""" …",null,1704085200,5.6,0.0,1015.7,0.0,0.0,"""2-3_oktas""",0.0,null,0,0,16093,null,"""DCA→11057""",18.570115,1,1,1,1,1,null,null,null,null
2024-01-01 00:00:00,"""AA""",153,"""AA""",19805,"""N894NN""",153,11278,30852,"""DCA""",11057,500,456,-4.0,-1,10.0,506,605,6.0,638,611,-27.0,0.0,null,0.0,98.0,75.0,59.0,1.0,331.0,null,null,null,null,null,null,0,…,"""BOEING ""","""737-823""","""5""","""5 ""","""1""","""0""","""02""","""162""","""CLASS 3""","""0000""",""" """,""" …",null,1704085200,5.6,0.0,1015.7,0.0,0.0,"""2-3_oktas""",0.0,null,0,0,16093,null,"""DCA→11057""",18.570115,1,1,1,1,1,null,null,null,null
2024-01-01 00:00:00,"""AA""",153,"""AA""",19805,"""N894NN""",153,11278,30852,"""DCA""",11057,500,456,-4.0,-1,10.0,506,605,6.0,638,611,-27.0,0.0,null,0.0,98.0,75.0,59.0,1.0,331.0,null,null,null,null,null,null,0,…,"""BOEING ""","""737-823""","""5""","""5 ""","""1""","""0""","""02""","""162""","""CLASS 3""","""0000""",""" """,""" …",null,1704085200,5.6,0.0,1015.7,0.0,0.0,"""2-3_oktas""",0.0,null,0,0,16093,null,"""DCA→11057""",18.570115,1,1,1,1,1,null,null,null,null
2024-01-01 00:00:00,"""AA""",153,"""AA""",19805,"""N894NN""",153,11278,30852,"""DCA""",11057,500,456,-4.0,-1,10.0,506,605,6.0,638,611,-27.0,0.0,null,0.0,98.0,75.0,59.0,1.0,331.0,null,null,null,null,null,null,0,…,"""BOEING ""","""737-823""","""5""","""5 ""","""1""","""0""","""02""","""162""","""CLASS 3""","""0000""",""" """,""" …",null,1704085200,5.6,0.0,1015.7,0.0,0.0,"""2-3_oktas""",0.0,null,0,0,16093,null,"""DCA→11057""",18.570115,1,1,1,1,1,null,null,null,null


In [14]:
# Copy dataset with cancelled flights dropped, to be used for regression model training (delay prediction)
non_cancelled_df = df.select(pl.all()).filter(pl.col("CANCELLED") == 0)
non_cancelled_df.shape, non_cancelled_df.head()

((8424706, 84),
 shape: (5, 84)
 ┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
 │ FL_DATE   ┆ MKT_UNIQU ┆ MKT_CARRI ┆ OP_UNIQUE ┆ … ┆ carrier_a ┆ carrier_c ┆ carrier_a ┆ carrier_ │
 │ ---       ┆ E_CARRIER ┆ ER_FL_NUM ┆ _CARRIER  ┆   ┆ vg_delay_ ┆ ancel_rat ┆ vg_delay_ ┆ cancel_r │
 │ datetime[ ┆ ---       ┆ ---       ┆ ---       ┆   ┆ week      ┆ e_week    ┆ month     ┆ ate_mont │
 │ μs]       ┆ cat       ┆ i64       ┆ cat       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ h        │
 │           ┆           ┆           ┆           ┆   ┆ f64       ┆ f64       ┆ f64       ┆ ---      │
 │           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ f64      │
 ╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
 │ 2024-01-0 ┆ AA        ┆ 153       ┆ AA        ┆ … ┆ null      ┆ null      ┆ null      ┆ null     │
 │ 1         ┆           ┆           ┆           ┆

## Prepare the data for model training

In [ ]:
# Extract features and target variable for both classification and regression models
cols = [
    # Flight info
    "scheduled_departure_ts",
    "MKT_UNIQUE_CARRIER",
    "MKT_CARRIER_FL_NUM",
    "DEST_AIRPORT_ID",
    "CRS_DEP_TIME",
    "DISTANCE",
    "MODEL",
    "avg_rte_delay",
    "deps_last_1h",
    "deps_last_3h",
    "deps_last_6h",
    "deps_last_12h",
    "deps_last_24h",
    "carrier_avg_delay_week",
    "carrier_avg_delay_month",
    "carrier_cancel_rate_month",
    "carrier_cancel_rate_week",
    # Weather info
    "temp",
    "dewtemp",
    "pressure",
    "winddirection",
    "windspeed",
    "skycoverage",
    "precipitation-1h",
    "precipitation-6h",
    "precip-1h_trace",
    "precip-6h_trace",
    "visibility_m",
    "ceiling_height_m",
]
x = non_cancelled_df.select(cols)
y = (
    non_cancelled_df.get_column("DEP_DELAY")
    .fill_nan(0.0)
    .fill_null(0.0)
    .clip(0.0, None)
)  # Replace NaN/Null delays with 0, since those likely indicate no delay; convert negative values to 0 (no delay)

x_cancel = df.select(cols)
y_cancel = df.select("CANCELLED")
x.shape, y.shape, x_cancel.shape, y_cancel.shape

((8424706, 25), (8424706,), (8674093, 25), (8674093, 1))

In [16]:
# Cancelled dataset
x_cancel = x_cancel.select(pl.all().exclude("CANCELLED"))
x_cancel.head(), x_cancel.shape, y_cancel.shape

(shape: (5, 25)
 ┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
 │ scheduled ┆ MKT_UNIQU ┆ MKT_CARRI ┆ DEST_AIRP ┆ … ┆ precip-1h ┆ precip-6h ┆ visibilit ┆ ceiling_ │
 │ _departur ┆ E_CARRIER ┆ ER_FL_NUM ┆ ORT_ID    ┆   ┆ _trace    ┆ _trace    ┆ y_m       ┆ height_m │
 │ e_ts      ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
 │ ---       ┆ cat       ┆ i64       ┆ i64       ┆   ┆ i32       ┆ i32       ┆ i32       ┆ i32      │
 │ i64       ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
 ╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
 │ 170408520 ┆ AA        ┆ 153       ┆ 11057     ┆ … ┆ 0         ┆ 0         ┆ 16093     ┆ null     │
 │ 0         ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
 │ 170408520 ┆ AA        ┆ 153       ┆ 11057     ┆ … ┆ 0         ┆

## Encode categorical features

In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder

categorical_cols = [col for col in x.columns if x[col].dtype == pl.Categorical]

# create column transformer -- https://stackoverflow.com/a/77965041
transformer = ColumnTransformer(
    transformers=[
        ("cat", OrdinalEncoder(), categorical_cols),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
)
transformer.set_output(transform="polars")

# Fit and transform the data
transformer.fit(df.select(cols))
x = transformer.transform(x)
x_cancel = transformer.transform(x_cancel)
x.head(), x_cancel.head()

AttributeError: 'list' object has no attribute 'get_loc'

In [ ]:
# Get the feature names after transformation, and the category-to-integer mappings for the categorical features
category_mappings = {
    col: {
        j: label
        for j, label in enumerate(transformer.named_transformers_["cat"].categories_[i])
    }
    for i, col in enumerate(categorical_cols)
}
category_mappings

## Train, validation, test split
We'll split the data first into validation and test sets. When we do hyperparameter tuning, we'll use the validation set with cross-validation to evaluate different hyperparameter combinations, and then we'll do a final evaluation on the test set to get the estimate of our model's performance on unseen data.

In [ ]:
from sklearn.model_selection import train_test_split

# Split the cancelled dataset
x_cancel_val, x_cancel_test, y_cancel_val, y_cancel_test = train_test_split(
    x_cancel, y_cancel, test_size=0.2, shuffle=False
)
x_cancel_val.shape, x_cancel_test.shape, y_cancel_val.shape, y_cancel_test.shape

In [ ]:
# Split the non-cancelled dataset
X_val, X_test, y_val, y_test = train_test_split(x, y, test_size=0.2, shuffle=False)
X_val.shape, X_test.shape, y_val.shape, y_test.shape

# Train the model
## Train a Random Forest Classifier for cancellation prediction

In [ ]:
from scipy.stats import randint
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

# Random forest for cancellation prediction
rf_cancel = RandomForestClassifier(random_state=42)

# Hyperparameter tuning
clf_param_dist = {
    "n_estimators": randint(200, 1001),
    "max_depth": [None, 10, 20, 30, 40, 50, 60],
    "min_samples_split": randint(2, 21),
    "min_samples_leaf": randint(1, 9),
    "max_features": ["sqrt", "log2", 0.3, 0.5, 0.7, 1.0],
    "bootstrap": [True, False],
    "class_weight": [None, "balanced"],
}
rfc_random = RandomizedSearchCV(
    rf_cancel,
    param_distributions=clf_param_dist,
    n_iter=HYPERPARAM_ITERS,
    cv=TimeSeriesSplit(n_splits=5),
    scoring="roc_auc",
    n_jobs=-1,
    verbose=2,
    random_state=42,
)
search = rfc_random.fit(x_cancel_val.to_numpy(), y_cancel_val.to_numpy().ravel())
print("Best Hyperparameters:", search.best_params_)

In [ ]:
rfc_random.score(x_cancel_test.to_numpy(), y_cancel_test.to_numpy().ravel())

In [ ]:
# Table of predicted probabilities and actual labels
y_cancel_pred = rfc_random.predict_proba(x_cancel_test.to_numpy())[:, 1]
cancel_df = pl.DataFrame(
    {
        "predicted_cancel_prob": y_cancel_pred,
        "actual_cancelled": y_cancel_test.to_numpy().ravel(),
    }
)
# treat actual as boolean
cancel_df = cancel_df.with_columns(pl.col("actual_cancelled").cast(pl.Boolean))
cancel_df.sample(20)

## Train a Random Forest Regressor for delay prediction

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Regress
regressor = RandomForestRegressor(random_state=42, oob_score=True)

# Hyperparameter tuning
rfr_param_dist = {
    "n_estimators": randint(200, 1001),
    "max_depth": [None, 10, 20, 30, 40, 50, 60],
    "min_samples_split": randint(2, 21),
    "min_samples_leaf": randint(1, 9),
    "max_features": ["sqrt", "log2", 0.3, 0.5, 0.7, 1.0],
    "bootstrap": [True],
}
rfr_random = RandomizedSearchCV(
    regressor,
    param_distributions=rfr_param_dist,
    n_iter=HYPERPARAM_ITERS,
    cv=TimeSeriesSplit(n_splits=5),
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=2,
    random_state=42,
)
regress_search = rfr_random.fit(X_val, y_val)
print("Best Hyperparameters:", regress_search.best_params_)

# Evaluation

In [ ]:
importances = regress_search.best_estimator_.feature_importances_
feature_names = X_val.columns

importance_df = pl.DataFrame(
    {
        "feature": feature_names,
        "importance": importances,
    }
).sort("importance", descending=True)

importance_df.show(None)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

# Evaluate
print("Out-of-Bag Score:", regress_search.best_estimator_.oob_score_)

y_pred = regress_search.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error:", mse)

r2 = r2_score(y_test, y_pred)
print("R-squared:", r2)

In [ ]:
import random

rand_start = random.randint(0, len(y_test) - 20)
print(f"{'Actual':>8} {'Predicted':>10} {'Difference':>11}")
for actual, predicted in zip(
    y_test[rand_start : rand_start + 20], y_pred[rand_start : rand_start + 20]
):
    print(f"{actual:8.2f} {predicted:10.2f} {actual - predicted:11.2f}")

# Save the trained models for later use

In [ ]:
# Save models with ONNX -- https://scikit-learn.org/stable/model_persistence.html#onnx
import json
from pathlib import Path

if EXPORT:
    # cancel_onnx = to_onnx(
    #     rfc_random.best_estimator_,
    #     X=x_cancel_val.to_numpy().astype("float32"),
    # )
    # regress_onnx = to_onnx(
    #     rfr_random.best_estimator_,
    #     X=X_val.to_numpy().astype("float32"),
    # )

    models_dir = Path("models")
    models_dir.mkdir(parents=True, exist_ok=True)

    # with open("models/cancel_model.onnx", "wb") as f:
    #     f.write(cancel_onnx.SerializeToString())
    # with open("models/regress_model.onnx", "wb") as f:
    #     f.write(regress_onnx.SerializeToString())

    metadata = {
        "feature_columns": cols,
        "classifier_features": x_cancel_val.columns,
        "regressor_features": X_val.columns,
        "categorical_columns": categorical_cols,
        "categorical_mappings": category_mappings,
        "classifier_best_params": rfc_random.best_params_,
        "regressor_best_params": rfr_random.best_params_,
    }
    with open(models_dir / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2, default=str)